# 🧠 Task 5: Mental Health Support Chatbot
**DevelopersHub Corporation — AI/ML Engineering Internship**

---

## 📌 Problem Statement
Build a chatbot that provides **supportive and empathetic responses** for people experiencing stress, anxiety, and emotional difficulties.

## 🎯 Goal
- Fine-tune a small language model (DistilGPT2) on the EmpatheticDialogues dataset
- Train it to respond empathetically using real human dialogues
- Build a command-line interactive interface to test the chatbot

## 📦 Dataset
**EmpatheticDialogues** by Facebook AI — a dataset of 25k conversations where speakers discuss emotional situations and receive empathetic responses.

## 🤖 Model
**DistilGPT2** — a lightweight, distilled version of GPT-2, suitable for fine-tuning in Colab.

> ⚠️ **Colab Tip:** Enable GPU — Runtime → Change runtime type → T4 GPU

---
## 1️⃣ Install Required Libraries

In [ ]:
# Install required libraries
!pip install transformers datasets accelerate --quiet
print('✅ Libraries installed!')

In [ ]:
# Core imports
import os
import math
import torch
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Using device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️ GPU not found. Enable it: Runtime → Change runtime type → T4 GPU')
    print('   Training will still work on CPU, just slower.')

---
## 2️⃣ Load & Explore the EmpatheticDialogues Dataset

In [ ]:
# Load EmpatheticDialogues dataset from Hugging Face
print('📥 Loading EmpatheticDialogues dataset...')
dataset = load_dataset('empathetic_dialogues')

print('✅ Dataset loaded!')
print(f'   Train samples: {len(dataset["train"])}')
print(f'   Valid samples: {len(dataset["validation"])}')
print(f'   Test  samples: {len(dataset["test"])}')

In [ ]:
# Inspect dataset structure
print('📋 Dataset Features:')
print(dataset['train'].features)
print('\n🔍 Sample entry:')
sample = dataset['train'][0]
for key, val in sample.items():
    print(f'  {key}: {val}')

In [ ]:
# Convert to pandas for EDA
train_df = pd.DataFrame(dataset['train'])

print('📊 Dataset Shape:', train_df.shape)
print('\n📋 Columns:', train_df.columns.tolist())
train_df.head()

In [ ]:
# Top emotions in the dataset
emotion_counts = train_df['context'].value_counts().head(20)

plt.figure(figsize=(14, 6))
bars = plt.barh(emotion_counts.index[::-1], emotion_counts.values[::-1],
                color='mediumslateblue', edgecolor='black', alpha=0.85)
plt.bar_label(bars, padding=3)
plt.xlabel('Number of Conversations', fontsize=12)
plt.title('🎭 Top 20 Emotions in EmpatheticDialogues', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📌 Total unique emotions: {train_df["context"].nunique()}')

In [ ]:
# Utterance length distribution
train_df['utterance_length'] = train_df['utterance'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['utterance_length'], bins=60,
             color='mediumslateblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Utterance Lengths', fontweight='bold')
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train_df['utterance_length'].median(), color='red',
                linestyle='--', label=f'Median: {train_df["utterance_length"].median():.0f}')
axes[0].legend()

# Show sample conversations
axes[1].axis('off')
sample_text = '\n'.join([
    'SAMPLE CONVERSATIONS',
    '─' * 35,
    f'Emotion: {train_df["context"][0]}',
    f'Speaker: {train_df["utterance"][0][:80]}...',
    '',
    f'Emotion: {train_df["context"][5]}',
    f'Speaker: {train_df["utterance"][5][:80]}...',
])
axes[1].text(0.05, 0.5, sample_text, transform=axes[1].transAxes,
             fontsize=10, verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('EmpatheticDialogues EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3️⃣ Preprocessing — Prepare Data for Fine-Tuning

In [ ]:
# Model name — DistilGPT2 is compact and Colab-friendly
MODEL_NAME = 'distilgpt2'
MAX_LENGTH = 128   # max tokens per sequence

# Load tokenizer
print(f'📥 Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT2 doesn't have a pad token by default — use EOS token
tokenizer.pad_token = tokenizer.eos_token

print('✅ Tokenizer loaded!')
print(f'   Vocab size:   {tokenizer.vocab_size:,}')
print(f'   Pad token:    {tokenizer.pad_token}')

In [ ]:
# Format conversations as: <emotion> [SEP] utterance <EOS>
# This teaches the model to respond in an emotional context
def format_conversation(example):
    """
    Format each dialogue turn as:
    'Emotion: <emotion>\nUser: <utterance>\n'
    This gives the model context about the emotional state.
    """
    emotion   = example.get('context', 'neutral')
    utterance = example.get('utterance', '')
    text = f"Emotion: {emotion}\nUser: {utterance}\nBot: I understand how you feel. "
    return {'text': text}

# Apply formatting
print('🔧 Formatting dataset...')
formatted_train = dataset['train'].map(format_conversation)
formatted_valid = dataset['validation'].map(format_conversation)

# Show formatted example
print('\n📝 Formatted Example:')
print(formatted_train[0]['text'])

In [ ]:
# Tokenization function
def tokenize_function(examples):
    """
    Tokenize text and truncate/pad to MAX_LENGTH.
    For causal language modeling, labels = input_ids (predict next token).
    """
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )
    result['labels'] = result['input_ids'].copy()
    return result

# Apply tokenization
print('🔢 Tokenizing dataset...')
tokenized_train = formatted_train.map(
    tokenize_function, batched=True,
    remove_columns=formatted_train.column_names
)
tokenized_valid = formatted_valid.map(
    tokenize_function, batched=True,
    remove_columns=formatted_valid.column_names
)

# Use subset for faster training (remove for full training)
# Use 5000 train + 1000 valid for a reasonable demo
tokenized_train = tokenized_train.select(range(min(5000, len(tokenized_train))))
tokenized_valid = tokenized_valid.select(range(min(1000, len(tokenized_valid))))

print(f'✅ Tokenization done!')
print(f'   Train samples: {len(tokenized_train)}')
print(f'   Valid samples: {len(tokenized_valid)}')

---
## 4️⃣ Load Model & Configure Training

In [ ]:
# Load DistilGPT2 model
print(f'📥 Loading model: {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Resize token embeddings (since we set pad token)
model.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model loaded!')
print(f'   Total parameters:     {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')

In [ ]:
# Data collator for Causal Language Modeling (CLM)
# mlm=False means we do next-token prediction, not masked LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not Masked LM
)

# Training arguments
training_args = TrainingArguments(
    output_dir='./mental_health_chatbot',        # Save directory
    num_train_epochs=3,                           # 3 epochs
    per_device_train_batch_size=8,                # batch size per GPU
    per_device_eval_batch_size=8,
    warmup_steps=100,                             # lr warmup
    weight_decay=0.01,                            # regularization
    learning_rate=5e-5,                           # fine-tuning LR
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy='epoch',                  # evaluate each epoch
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),               # use fp16 on GPU
    report_to='none',                             # disable wandb
    dataloader_pin_memory=False
)

print('✅ Training arguments configured!')
print(f'   Epochs:      {training_args.num_train_epochs}')
print(f'   Batch size:  {training_args.per_device_train_batch_size}')
print(f'   LR:          {training_args.learning_rate}')
print(f'   FP16:        {training_args.fp16}')

---
## 5️⃣ Fine-Tune the Model

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print('🚀 Starting fine-tuning...')
print('   This may take 5–15 minutes on GPU, longer on CPU.')
print('   Grab a chai! ☕\n')

# Train!
train_result = trainer.train()

print('\n✅ Fine-tuning complete!')
print(f'   Training loss:   {train_result.training_loss:.4f}')
print(f'   Training time:   {train_result.metrics["train_runtime"]:.0f} seconds')

In [ ]:
# Evaluate the model
print('📊 Evaluating model...')
eval_results = trainer.evaluate()

perplexity = math.exp(eval_results['eval_loss'])
print(f'\n📌 Evaluation Results:')
print(f'   Eval Loss:    {eval_results["eval_loss"]:.4f}')
print(f'   Perplexity:   {perplexity:.2f}')
print()
print('📖 Perplexity Interpretation:')
print('   Lower perplexity = model is more confident/accurate.')
print('   Starting perplexity for DistilGPT2 is typically ~20-40.')
print('   After fine-tuning on empathetic data, it should decrease.')

In [ ]:
# Save the fine-tuned model
SAVE_PATH = './mental_health_chatbot_final'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'✅ Model saved to: {SAVE_PATH}')

---
## 6️⃣ Training Loss Visualization

In [ ]:
# Extract training history
log_history = trainer.state.log_history

# Separate train loss and eval loss
train_losses = [(entry['step'], entry['loss'])
                for entry in log_history if 'loss' in entry and 'eval_loss' not in entry]
eval_losses  = [(entry['step'], entry['eval_loss'])
                for entry in log_history if 'eval_loss' in entry]

if train_losses:
    steps_t, losses_t = zip(*train_losses)
    steps_e, losses_e = zip(*eval_losses) if eval_losses else ([], [])

    plt.figure(figsize=(12, 5))
    plt.plot(steps_t, losses_t, label='Training Loss',   color='steelblue',  linewidth=2)
    if steps_e:
        plt.plot(steps_e, losses_e, label='Validation Loss', color='darkorange',
                 linewidth=2, marker='o', markersize=8)

    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('🧠 Fine-Tuning Loss Curve — Mental Health Chatbot', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    print('📌 A decreasing loss curve indicates successful learning!')
else:
    print('ℹ️ No training logs to plot yet — run training first.')

---
## 7️⃣ Generate Empathetic Responses — Test the Chatbot

In [ ]:
# Load fine-tuned model for inference
print('📥 Loading fine-tuned model for chat...')
chat_model = AutoModelForCausalLM.from_pretrained(SAVE_PATH)
chat_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
chat_model.eval()

if device == 'cuda':
    chat_model = chat_model.to('cuda')

print('✅ Model ready for conversation!')

In [ ]:
# Safety keywords — avoid generating harmful content
HARMFUL_KEYWORDS = [
    'suicide', 'kill yourself', 'hurt yourself',
    'end your life', 'self-harm', 'overdose'
]

CRISIS_RESPONSE = (
    "I can hear that you're going through a really difficult time. "
    "Please reach out to a professional who can help. "
    "If you're in crisis, please call or text 988 (Suicide & Crisis Lifeline). "
    "You are not alone, and help is available. 💙"
)

def generate_response(user_input, emotion='sad', max_new_tokens=80):
    """
    Generate an empathetic response for the given user input.
    Includes safety filtering for harmful content.
    """
    # ---- SAFETY CHECK ----
    input_lower = user_input.lower()
    if any(keyword in input_lower for keyword in HARMFUL_KEYWORDS):
        return CRISIS_RESPONSE

    # ---- FORMAT PROMPT ----
    prompt = f"Emotion: {emotion}\nUser: {user_input}\nBot: I understand how you feel. "

    # ---- TOKENIZE ----
    inputs = chat_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=100
    )
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

    # ---- GENERATE ----
    with torch.no_grad():
        outputs = chat_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,        # creativity level
            top_p=0.92,             # nucleus sampling
            repetition_penalty=1.3, # reduce repetition
            pad_token_id=chat_tokenizer.eos_token_id,
            eos_token_id=chat_tokenizer.eos_token_id
        )

    # ---- DECODE ----
    full_text = chat_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the Bot's response
    if 'Bot:' in full_text:
        response = full_text.split('Bot:')[-1].strip()
    else:
        response = full_text[len(prompt):].strip()

    # Trim to first complete sentence if possible
    for punct in ['.', '!', '?']:
        if punct in response:
            response = response[:response.rfind(punct)+1]
            break

    return response if response.strip() else "I'm here for you. Tell me more about how you're feeling."

print('✅ Response generator ready!')

In [ ]:
# ---- TEST THE CHATBOT ----
test_cases = [
    ('I feel so stressed about my exams, I can\'t sleep.',          'anxious'),
    ('Nobody understands me. I feel completely alone.',              'lonely'),
    ('I\'ve been feeling really down lately and don\'t know why.', 'sad'),
    ('I\'m so angry. Everything is going wrong today.',             'angry'),
    ('I\'m proud of myself for finishing a hard project!',          'joyful'),
    ('I\'m worried about my future and career.',                    'worried'),
]

print('=' * 65)
print('      🧠 MENTAL HEALTH CHATBOT — TEST RESPONSES')
print('=' * 65)

for user_msg, emotion in test_cases:
    response = generate_response(user_msg, emotion=emotion)
    print(f'\n🎭 Emotion: {emotion.upper()}')
    print(f'👤 User:    {user_msg}')
    print(f'🤖 Bot:     {response}')
    print('-' * 65)

# Safety filter test
print('\n🛡️ SAFETY FILTER TEST:')
safe_test = generate_response('I want to hurt myself', emotion='sad')
print(f'🤖 Bot: {safe_test}')

---
## 8️⃣ Interactive Command-Line Chatbot

In [ ]:
# ============================================================
# INTERACTIVE CHATBOT — Run this cell to chat with the bot!
# Type 'quit' or 'exit' to stop.
# ============================================================

EMOTION_MAP = {
    'stress': 'anxious', 'anxiety': 'anxious', 'anxious': 'anxious',
    'sad': 'sad', 'depressed': 'sad', 'cry': 'sad', 'crying': 'sad',
    'angry': 'angry', 'anger': 'angry', 'frustrated': 'angry',
    'happy': 'joyful', 'excited': 'joyful', 'proud': 'joyful',
    'lonely': 'lonely', 'alone': 'lonely', 'isolated': 'lonely',
    'worried': 'worried', 'scared': 'worried', 'fear': 'worried',
}

def detect_emotion(text):
    """Simple keyword-based emotion detection."""
    text_lower = text.lower()
    for keyword, emotion in EMOTION_MAP.items():
        if keyword in text_lower:
            return emotion
    return 'sad'  # default emotion

print('=' * 60)
print('  🧠 Mental Health Support Chatbot — Interactive Mode')
print('=' * 60)
print('  💙 I am here to listen and support you.')
print('  Type your message below. Type "quit" to exit.')
print('=' * 60)

conversation_count = 0
MAX_TURNS = 10  # limit for notebook demo

while conversation_count < MAX_TURNS:
    try:
        user_input = input('\n👤 You: ').strip()
    except EOFError:
        break

    if not user_input:
        continue

    if user_input.lower() in ['quit', 'exit', 'bye', 'goodbye']:
        print('\n🤖 Bot: Take care of yourself. Remember, you are never alone. 💙')
        print('        Goodbye! Come back anytime you need support.')
        break

    # Detect emotion from input
    emotion = detect_emotion(user_input)

    # Generate response
    response = generate_response(user_input, emotion=emotion)

    print(f'\n🤖 Bot: {response}')
    print(f'   [Detected emotion: {emotion}]')

    conversation_count += 1

if conversation_count >= MAX_TURNS:
    print(f'\n[ℹ️ Demo limit reached ({MAX_TURNS} turns). Restart cell to chat again.]')

---
## 9️⃣ Final Summary & Insights

In [ ]:
print('=' * 65)
print('    🧠 MENTAL HEALTH CHATBOT — FINAL SUMMARY')
print('=' * 65)
print('''
📌 Task:       Mental Health Support Chatbot
📌 Model:      DistilGPT2 (fine-tuned)
📌 Dataset:    EmpatheticDialogues (Facebook AI)
               25,000+ empathetic conversations, 32 emotions
📌 Epochs:     3
📌 Approach:   Causal Language Modeling (next token prediction)

🔧 What Was Done:
   1. Loaded & explored EmpatheticDialogues dataset
   2. Formatted dialogues with emotion context
   3. Fine-tuned DistilGPT2 using HuggingFace Trainer API
   4. Built emotion detection from keywords
   5. Added safety filters for harmful content
   6. Created interactive command-line chatbot

🛡️ Safety Features:
   • Detects harmful keywords → returns crisis resources
   • Never encourages self-harm
   • Redirects to professional help (988 Lifeline)

📊 Key Insights:
   • Fine-tuning on emotional dialogues shifts the model's
     tone to be warmer and more empathetic
   • DistilGPT2 is fast but limited — larger models like
     GPT-Neo or Mistral-7B would produce better quality
   • Perplexity decreased after fine-tuning, showing the
     model learned domain-specific language patterns
   • Emotion-conditioned prompts help guide response tone
''')
print('=' * 65)